# MeMo HF
Version integrated with Transformer Libraries (Version 0.4)

In [22]:
import torch
from MeMoHF.modelling_memo_tokenizer import MeMoTokenizer
from MeMoHF.modelling_memo_configuration import MeMoConfig
from MeMoHF.modelling_memo import MeMoForCausalLM
from MeMoHF.evaluating_memo import Evaluation, EvaluationUpdateNew

In [23]:
max_length = 12 
tokenizer = MeMoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b", 
                                          truncation_side = 'left',
                                          padding_side='left', model_max_length=max_length, head_number=4)
tokenizer.save_pretrained('tok_exp')
tokenizer = MeMoTokenizer.from_pretrained('tok_exp')
tokenizer

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPTNeoXTokenizer'. 
The class this function is called from is 'MeMoTokenizer'.


Setting pad token and pad token id = <|endoftext|>, 0
Setting pad token and pad token id = <|endoftext|>, 0


MeMoTokenizer(name_or_path='tok_exp', vocab_size=50254, model_max_length=13, is_fast=True, padding_side='left', truncation_side='left', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<|padding|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50254: AddedToken("                        ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50255: AddedToken("                       ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50256: AddedToken("                      ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50257: AddedToken("                     ", rstrip=False, lstrip=Fal

In [24]:
with open("testo_di_prova.txt") as my_first_text_f:
    my_first_text = my_first_text_f.read()

token_ids = tokenizer.encode(my_first_text)#, return_tensors='pt')
print(token_ids) # return max len + 1 

(tensor([[18886,   256, 36144,  4164,  1809,    80,  1448,   295,   532,  1584,
            13, 50190]]), tensor([[  256, 36144,  4164,  1809,    80,  1448,   295,   532,  1584,    13,
         50190,    15]]))


In [25]:
memo_input = tokenizer.get_text_batch_encoding([my_first_text, my_first_text[0:10]])
memo_input.keys(), memo_input['input_ids'].shape

Token indices sequence length is longer than the specified maximum sequence length for this model (654 > 13). Running this sequence through the model will result in indexing errors


(dict_keys(['input_ids', 'labels']), torch.Size([52, 12]))

In [26]:
for i in range(3):
    print(tokenizer.decode(memo_input['input_ids'][i]))
    print(tokenizer.decode(memo_input['labels'][i]))
    print()

<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>Cosimo di
<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>Cosimo di Giovanni

 de' Medici detto il Vecchio o Pater
' Medici detto il Vecchio o Pater patri

æ (Firenze, 27 settembre 1389
 (Firenze, 27 settembre 1389 –



Check memorization on single layer

In [27]:
memo_input = tokenizer.get_text_batch_encoding([my_first_text, my_first_text[10:30]])

memo_input['input_ids'].shape

torch.Size([52, 12])

In [28]:
from MeMoHF.modelling_memo_embedding import MeMoEmbedding

In [29]:
d,h,l = 1024, 4, 3

In [30]:
embedding = MeMoEmbedding(
    num_embeddings=len(tokenizer),# tokenizer.vocab_size,
    embedding_dim=d,
    padding_idx=tokenizer.pad_token_id, #0
    _freeze=True
)

MeMo embedding initilialization


In [31]:
input_embeddings = embedding.encode(memo_input['input_ids'])
output_symbols = embedding.encode(memo_input['labels'])

input_embeddings.shape, output_symbols.shape

(torch.Size([52, 12, 1024]), torch.Size([52, 12, 1024]))

In [32]:
input_tokens_ids = tokenizer(['Test', 'Un altro Test'])['input_ids']
print(input_tokens_ids)

input_embeddings = embedding.forward(input_tokens_ids)
input_embeddings

tensor([[   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
         5089],
        [   0,    0,    0,    0,    0,    0,    0,    0,    0, 2447, 6945,  287,
         6004]])


tensor([[[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         ...,
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [-0.0303,  0.0379, -0.0315,  ..., -0.0358, -0.0159,  0.0667]],

        [[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         ...,
         [ 0.0755,  0.0051, -0.0254,  ...,  0.0273,  0.0307, -0.0528],
         [ 0.0045, -0.0443,  0.0026,  ..., -0.0521,  0.0484, -0.0168],
         [-0.0946,  0.0153,  0.0372,  ..., -0.0027,  0.0205, -0.0025]]])

In [33]:
from MeMoHF.modelling_memo_layer import MeMoLayer

In [34]:
layer = MeMoLayer(d, h)
layer

MeMoLayer(
  (W_v_single_head): ProjectionTokens(in_features=1024, out_features=256)
  (Prj): ProjectionSequence((trasposed wrt saved one) in_features=4096, out_features=1024)
)

Memo: Initializing the Tokenizer and the model

In [35]:
# Meta Parameters : 
#    d - inner dimension
#    h - number of heads
#    l - number of layers
d,h,l = 1024, 4, 4
chunk_length = 1024

# Initializing a standard Tokenizer
max_length = chunk_length 
tokenizer = MeMoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b", 
                                          padding_side='left', truncation_side='left', 
                                          model_max_length=max_length, head_number=h)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.pad_token_id

# Intializing Memo Configuration
config = MeMoConfig(vocab_size=len(tokenizer), #tokenizer.vocab_size, 
               hidden_size=d, 
               num_hidden_layers=l,
               num_attention_heads=h,
               chunk_length=chunk_length,
               bos_token_id=tokenizer.bos_token_id,
               eos_token_id=tokenizer.eos_token_id,
               pad_token_id=tokenizer.pad_token_id,
              )

# Initializing the Memo Model from the configuration

model = MeMoForCausalLM(config) 

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    model.to('cuda')


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPTNeoXTokenizer'. 
The class this function is called from is 'MeMoTokenizer'.


Setting pad token and pad token id = <|endoftext|>, 0
MeMo embedding initilialization
MeMo embedding initilialization
MeMo embedding initilialization
GPU: NVIDIA GeForce RTX 3080 Ti Laptop GPU is available.


Reading the two texts

In [36]:
with open("testo_di_prova.txt") as my_first_text_f:
    my_first_text = my_first_text_f.read()
with open("testo_di_prova2.txt") as my_first_text_f:
    my_second_text = my_first_text_f.read()



In [37]:
# batch_inputs = tokenizer.get_text_batch_encoding_for_loss(text=[my_first_text])
# outs = model.forward_with_loss_parallelized(
#     batch_inputs=batch_inputs,
# )
# print(outs['loss'])

In [38]:
# memo_input_test = tokenizer.get_text_batch_encoding([my_first_text])
# model.memorize_text(memo_input_test)

In [39]:
# batch_inputs = tokenizer.get_text_batch_encoding_for_loss(text=my_first_text)
# outs = model.forward_with_loss(
#     batch_inputs=batch_inputs,
# )
# print(outs['loss'])

Memorizing the first text and evaluating if it is memorized

In [41]:
memo_input_1 = tokenizer.get_text_batch_encoding([my_first_text]*8)  # Writing the same doc 8 times to stress the memorization with batch
memo_input_2 = tokenizer.get_text_batch_encoding([my_second_text]*8) # Writing the same doc 8 times to stress the memorization with batch

model.memorize_text(memo_input_1)
e = EvaluationUpdateNew()

e1 = e.check_pretokenized(model, tokenizer, memo_input_1['input_ids'], starting_point=8)
e2 = e.check_pretokenized(model, tokenizer, memo_input_2['input_ids'], starting_point=8)

print("Memorization level of first text  : ", e1) 
print("Memorization level of second text : ", e2) 


e1 = e.check_pretokenized(model, tokenizer, memo_input_1['input_ids'], starting_point=0)
e2 = e.check_pretokenized(model, tokenizer, memo_input_2['input_ids'], starting_point=0)

print("Memorization level of first text  : ", e1) 
print("Memorization level of second text : ", e2) 

Starting point : 8


100%|██████████| 1015/1015 [00:01<00:00, 527.25it/s]


Starting point : 8


100%|██████████| 1015/1015 [00:02<00:00, 384.53it/s]


Memorization level of first text  :  tensor(0.9985)
Memorization level of second text :  tensor(0.0232)
Starting point : 0


100%|██████████| 1023/1023 [00:01<00:00, 572.62it/s]


Starting point : 0


100%|██████████| 1023/1023 [00:02<00:00, 372.88it/s]


Memorization level of first text  :  tensor(0.9985)
Memorization level of second text :  tensor(0.0230)


In [ ]:
batch_inputs = tokenizer.get_text_batch_encoding_for_loss(text=[my_first_text]*8)
with torch.no_grad():
    model.eval()
    outs = model.forward_with_loss_parallelized(
        batch_inputs=batch_inputs
    )
    model.train()
loss = outs['loss']
del outs
print(loss)

OutOfMemoryError: CUDA out of memory. Tried to allocate 412.00 MiB. GPU 0 has a total capacity of 15.62 GiB of which 397.69 MiB is free. Including non-PyTorch memory, this process has 15.13 GiB memory in use. Of the allocated memory 14.88 GiB is allocated by PyTorch, and 25.94 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

Memorizing the second text and checking if it affected the memorization of the first text

In [ ]:
model.memorize_text(memo_input_2)

e1 = e.check_pretokenized(model, tokenizer, memo_input_1['input_ids'], starting_point=8)
e2 = e.check_pretokenized(model, tokenizer, memo_input_2['input_ids'], starting_point=8)

print("Memorization level of first text  : ", e1) 
print("Memorization level of second text : ", e2) 

Forgetting the first document

In [ ]:
model.forget_text(memo_input_2)

Checking the effect on the two texts

In [ ]:
e1 = e.check_pretokenized(model, tokenizer, memo_input_1['input_ids'], starting_point=8)
e2 = e.check_pretokenized(model, tokenizer, memo_input_2['input_ids'], starting_point=8)

print("Memorization level of first text  : ", e1) 
print("Memorization level of second text : ", e2) 

In [ ]:
model

In [ ]:
with open("testo_di_prova2.txt") as my_first_text_f:
    my_second_text = my_first_text_f.read()


In [ ]:
config = MeMoConfig(vocab_size=len(tokenizer), #tokenizer.vocab_size, 
               hidden_size=d, 
               num_hidden_layers=l,
               num_attention_heads=h,
               chunk_length=chunk_length,
               bos_token_id=tokenizer.bos_token_id,
               eos_token_id=tokenizer.eos_token_id,
               pad_token_id=tokenizer.pad_token_id,
              )

# Initializing the Memo Model from the configuration

model = MeMoForCausalLM(config)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    model.to('cuda')
# print("CMM pre learning")
# display(model.memo.layers[0].CMM.weight)


bs = 8
for b in range(bs):
    print(f"memorizing the same text iteration = {b}")
    memo_input = tokenizer.get_text_batch_encoding(my_first_text)
    model.memorize_text(memo_input)

# Prj = model.memo.layers[0].Prj.weight.detach().cpu()
# CMM = model.memo.layers[0].CMM.weight.detach().cpu()

# display(Prj.T @ Prj)
# display(CMM)

e = Evaluation() #UpdateNew()
out = e.check_pretokenized(model, tokenizer, memo_input['input_ids'])
print("Degree of memorization after memorizing 1: %f ", out)


memo_input_2 = tokenizer.get_text_batch_encoding(my_second_text) 
model.memorize_text(memo_input_2)
out = e.check_pretokenized(model, tokenizer, memo_input['input_ids'])
print("Degree of memorization of test 1 after memorizing 2: %f ", out)
out = e.check_pretokenized(model, tokenizer, memo_input_2['input_ids'])
print("Degree of memorization of test 2 after memorizing 2: %f ", out)


for b in range(bs):
    print(f"forgetting the same text iteration = {b}")
    memo_input = tokenizer.get_text_batch_encoding(my_first_text)
    model.forget_text(memo_input)

out = e.check_pretokenized(model, tokenizer, memo_input['input_ids'])
print("Degree of memorization of test 1 after forgetting 1: %f ", out)
out = e.check_pretokenized(model, tokenizer, memo_input_2['input_ids'])
print("Degree of memorization of test 2 after after forgetting 1: %f ", out)

In [ ]:
model.save_pretrained('MemoExp')
tokenizer.save_pretrained('MemoExp')

In [ ]:
# from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM
model1 = MeMoForCausalLM.from_pretrained("MemoExp", device_map="auto")
model = model1

In [ ]:
model.device

In [ ]:
model.config

In [ ]:
# code for testing generation capabilities of MeMo
model.eval()

memo_input_test = tokenizer.get_text_batch_encoding(my_first_text)
print(f'text={my_first_text}\nmemo_input={memo_input_test}')

generated_text = model.generate(inputs=memo_input_test['input_ids'].to('cuda'), max_new_tokens=20)

generated_text.shape
final_text = tokenizer.decode(generated_text[0], skip_special_token=True)
next_sequence = tokenizer.decode(generated_text[0][memo_input_test['input_ids'].shape[1]:], skip_special_tokens=True)
print(final_text)
print(next_sequence)

In [ ]:
batch_inputs = tokenizer.get_text_batch_encoding_for_loss(text=my_first_text)
outs = model.forward_with_loss(
    batch_inputs=batch_inputs,
)

In [ ]:
print(outs['loss'])